In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("store_data.csv")

In [3]:
df.head()

,shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil
0,burgers,meatballs,eggs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,chutney,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,turkey,avocado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,mineral water,milk,energy bar,whole wheat rice,green tea,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,low fat yogurt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df = df.head(500)

In [5]:
df.shape

(500, 20)

In [6]:
df['almonds'][0]

'meatballs'

In [7]:
transactions = []
for row in df.itertuples(index=False):
    c = []
    for item in row:
        if isinstance(item, str):
            #print(item)
            c.append(item)
    transactions.append(c)

In [8]:
unique_items = set()
for txn in transactions:
    for item in txn:
        unique_items.add(item)

In [9]:
l = []
for i in range(len(transactions)):
  row_dict = dict.fromkeys(unique_items, 0)
  for item in transactions[i]:
    row_dict[item] = 1
  l.append(row_dict)

In [10]:
l = pd.DataFrame(l)

In [11]:
l.head()

,zucchini,energy drink,bacon,pickles,cake,champagne,blueberries,tea,avocado,tomato juice,...,escalope,mint green tea,magazines,olive oil,cider,fresh bread,meatballs,chocolate bread,whole wheat rice,body spray
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
min_support = 0.04
f1 = {}
for i in l.columns:
  c=0
  for j in range(len(transactions)):
    if l[i][j]==1:
      c+=1
  if c/len(transactions)>=min_support:
    f1[i] = c/len(transactions)

In [13]:
f1

{'cake': 0.068,
 'champagne': 0.048,
 'tomato juice': 0.048,
 'shrimp': 0.076,
 'frozen smoothie': 0.052,
 'cooking oil': 0.048,
 'burgers': 0.078,
 'pasta': 0.046,
 'mineral water': 0.24,
 'eggs': 0.206,
 'chicken': 0.052,
 'turkey': 0.074,
 'red wine': 0.044,
 'french fries': 0.158,
 'spaghetti': 0.192,
 'milk': 0.112,
 'frozen vegetables': 0.086,
 'ground beef': 0.098,
 'green tea': 0.116,
 'salmon': 0.044,
 'pancakes': 0.06,
 'cookies': 0.08,
 'soup': 0.076,
 'tomatoes': 0.046,
 'low fat yogurt': 0.062,
 'chocolate': 0.18,
 'grated cheese': 0.04,
 'herb & pepper': 0.062,
 'escalope': 0.066,
 'olive oil': 0.058,
 'whole wheat rice': 0.042}

In [14]:
import pandas as pd
import itertools

# l = your one-hot encoded DataFrame
min_support = 0.1
num_transactions = len(l)

# -------------------------------------
# STEP 1: Generate L1 (frequent 1-itemsets)
# -------------------------------------

def get_L1(df, min_support):
    L1 = {}
    for item in df.columns:
        count = df[item].sum()              # number of rows with 1
        support = count / len(df)
        if support >= min_support:
            L1[frozenset([item])] = support
    return L1

L1 = get_L1(l, min_support)
frequent_itemsets = []      # to store L1, L2, L3, ...
frequent_itemsets.append(L1)

# -------------------------------------
# Helper: get support for any k-itemset
# -------------------------------------

def get_support(df, itemset):
    count = 0
    for row in range(len(df)):
        present = True
        for item in itemset:
            if df.loc[row, item] != 1:
                present = False
                break
        if present:
            count += 1
    return count / len(df)

# -------------------------------------
# Helper: Generate candidate itemsets from Lk
# -------------------------------------

def generate_candidates(Lk):
    items = list(Lk.keys())
    candidates = []

    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            a = items[i]
            b = items[j]

            # join if first k-1 elements are same
            union = a.union(b)
            if len(union) == len(a) + 1:
                candidates.append(union)

    # remove duplicates
    candidates = list(set(candidates))
    return candidates

# -------------------------------------
# Helper: prune candidates using Apriori property
# -------------------------------------

def prune(candidates, Lk):
    Lk_sets = list(Lk.keys())      # previous level frequent sets
    pruned = []

    for c in candidates:
        all_subsets_frequent = True

        for subset in itertools.combinations(c, len(c)-1):
            if frozenset(subset) not in Lk_sets:
                all_subsets_frequent = False
                break

        if all_subsets_frequent:
            pruned.append(c)

    return pruned

# -------------------------------------
# STEP K: Generate L2, L3, ... until no more itemsets
# -------------------------------------

Lk = L1

while True:
    # Generate candidate itemsets from Lk
    Ck = generate_candidates(Lk)

    # Prune candidates using Apriori rule
    Ck = prune(Ck, Lk)

    # Count support of each candidate
    Lk1 = {}
    for candidate in Ck:
        support = get_support(l, candidate)
        if support >= min_support:
            Lk1[candidate] = support

    # If no more frequent itemsets, stop
    if len(Lk1) == 0:
        break

    # Save this level's frequent itemsets
    frequent_itemsets.append(Lk1)

    # Move to next level
    Lk = Lk1


# -------------------------------------
# frequent_itemsets now contains:
# [L1, L2, L3, ...]
# -------------------------------------

# OPTIONAL: flatten into one dict
all_frequent = {}
for level in frequent_itemsets:
    all_frequent.update(level)

# all_frequent = {frozenset({...}): support}

In [15]:
import itertools

# frequent_itemsets = output from Step 4 (list of dicts)
# all_frequent = flattened dict {frozenset: support}

min_confidence = 0.0
min_lift = 0.0

# total transactions
num_transactions = len(l)

rules = []   # store rules as dicts: {'antecedent':..., 'consequent':..., 'support':..., 'confidence':..., 'lift':...

# ----------------------------------------
# STEP 5: Generate all association rules
# ----------------------------------------

for itemset, support_itemset in all_frequent.items():

    # Only consider itemsets of size >= 2 (you cannot split 1-itemsets)
    if len(itemset) < 2:
        continue

    # Generate ALL possible non-empty proper subsets
    # For an itemset {A,B,C}, subsets = {A}, {B}, {C}, {A,B}, {A,C}, {B,C}
    for r in range(1, len(itemset)):
        for antecedent in itertools.combinations(itemset, r):

            antecedent = frozenset(antecedent)
            consequent = itemset - antecedent

            # compute support(antecedent)
            support_ante = get_support(l, antecedent)

            # compute support(consequent)
            support_cons = get_support(l, consequent)

            # compute confidence = support(A∪B) / support(A)
            confidence = support_itemset / support_ante

            # compute lift = confidence / support(consequent)
            lift = confidence / support_cons

            # apply thresholds
            if confidence >= min_confidence and lift >= min_lift:
                rules.append({
                    "antecedent": antecedent,
                    "consequent": consequent,
                    "support": support_itemset,
                    "confidence": confidence,
                    "lift": lift
                })

In [16]:
rules

[]